# ⚽ FIFA World Cup 2026 Predictor

Runs the full prediction pipeline in Google Colab:

1. **Setup** — clone the repo and install dependencies
2. **Pipeline** — download ~48k matches, build features, train LightGBM models, simulate the tournament
3. **Results** — championship odds chart and table, inline
4. **Browser UI** *(optional)* — serve the interactive dashboard through Colab's port proxy

Just run the cells top to bottom (`Runtime → Run all`). The full pipeline takes a few minutes on the free tier.

## 1. Setup

In [ ]:
%cd /content
!rm -rf wm && git clone -q https://github.com/lukas04545/wm.git
%cd /content/wm
!pip install -q -e .
print('✓ Setup complete')

## 2. Run the pipeline

In [ ]:
!python -m wm.cli ingest --source results
!python -m wm.cli build-features
!python -m wm.cli train

In [ ]:
# Monte Carlo tournament simulation (increase --runs for tighter estimates)
!python -m wm.cli simulate --runs 20000

## 3. Results — championship odds

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

with open('reports/simulation/results.json') as f:
    sim = json.load(f)

rows = [
    {
        'Team': team,
        'Champion %': d.get('champion', 0) * 100,
        'Final %': d.get('final', 0) * 100,
        'Semifinal %': d.get('semifinal', 0) * 100,
        'Quarterfinal %': d.get('quarterfinal', 0) * 100,
        'Qualified %': d.get('group_qualified', 0) * 100,
        'Avg group pts': d.get('avg_group_points', 0),
    }
    for team, d in sim.items() if not team.startswith('_')
]
odds = pd.DataFrame(rows).sort_values('Champion %', ascending=False).reset_index(drop=True)
odds.index += 1

top16 = odds.head(16).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top16['Team'], top16['Champion %'], color=plt.cm.viridis_r(top16['Champion %'] / top16['Champion %'].max()))
ax.set_xlabel('Championship probability (%)')
ax.set_title('FIFA World Cup 2026 — Championship Odds')
for bar, v in zip(bars, top16['Champion %']):
    ax.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height() / 2, f'{v:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

odds.round(1)

In [ ]:
# Most likely finals
meta = sim.get('_meta', {})
print(f"Based on {meta.get('n_runs', '?'):,} simulated tournaments\n")
print('Most likely finals:')
for i, (teams, prob) in enumerate(meta.get('top_final_pairings', [])[:8], 1):
    print(f'  {i}. {teams[0]} vs {teams[1]}  —  {prob*100:.2f}%')

## 4. Predict any single match

In [ ]:
!python -m wm.cli predict "Argentina" "France" --venue "MetLife Stadium"
!python -m wm.cli predict "Brazil" "Germany"

## 5. Interactive browser UI *(optional)*

Serves the dashboard through Colab's built-in port proxy — click the link the cell prints.

In [ ]:
import subprocess, time
from google.colab.output import eval_js

server = subprocess.Popen(['python', '-m', 'wm.cli', 'serve', '--port', '8000'])
time.sleep(5)
url = eval_js('google.colab.kernel.proxyPort(8000)')
print(f'⚽ Open the dashboard: {url}')

In [ ]:
# Or embed the dashboard directly in the notebook:
from IPython.display import IFrame, display
display(IFrame(src=url, width='100%', height=800))